# Comprehensive Regression Analysis: XGBoost vs Linear Regression

## Overview
This notebook provides a comprehensive comparison between **XGBoost Regressor** and **Linear Regression** models, including:

- **Multiple Train-Test Split Strategies**
- **Three Hyperparameter Tuning Methods**: Grid Search, Random Search, and Bayesian Optimization
- **Linear Regression Assumption Testing**: Linearity, Independence, Homoscedasticity, Normality, and Multicollinearity
- **Advanced Visualizations**: Plotly interactive charts for model performance, gradient descent, and assumption verification
- **Model Persistence**: Saving models using both Pickle and Joblib

---

## Table of Contents
1. Import Libraries
2. Load and Explore Data
3. Data Preprocessing
4. Train-Test Split Strategies
5. **XGBoost Regressor**
   - Baseline Model
   - Grid Search CV
   - Random Search CV
   - Bayesian Optimization
   - Evaluation and Visualization
   - Model Saving
6. **Linear Regression**
   - Baseline Model
   - Assumption Testing
   - Hyperparameter Tuning (Regularization)
   - Gradient Descent Visualization
   - Evaluation and Visualization
   - Model Saving
7. Model Comparison

## 1. Import Required Libraries

### Library Categories:
- **Data Manipulation**: pandas, numpy
- **Visualization**: plotly (express, graph_objects, subplots)
- **Machine Learning Models**: scikit-learn (LinearRegression, Ridge, Lasso, ElasticNet), XGBoost
- **Preprocessing**: StandardScaler, MinMaxScaler
- **Model Selection**: train_test_split, GridSearchCV, RandomizedSearchCV, cross_val_score
- **Metrics**: mean_squared_error, mean_absolute_error, r2_score
- **Hyperparameter Optimization**: scikit-optimize (BayesSearchCV)
- **Statistical Testing**: scipy.stats, statsmodels
- **Model Persistence**: pickle, joblib
- **Utilities**: warnings

In [2]:
# Core Data Science Libraries
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Visualization
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Machine Learning Models
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from xgboost import XGBRegressor

# Preprocessing
from sklearn.preprocessing import StandardScaler, MinMaxScaler

# Model Selection and Evaluation
from sklearn.model_selection import (train_test_split, GridSearchCV, 
                                      RandomizedSearchCV, cross_val_score, 
                                      KFold, learning_curve)
from sklearn.metrics import (mean_squared_error, mean_absolute_error, 
                             r2_score, mean_absolute_percentage_error)

# Bayesian Optimization
from skopt import BayesSearchCV
from skopt.space import Real, Integer, Categorical

# Statistical Testing
from scipy import stats
from scipy.stats import shapiro, normaltest, anderson
from statsmodels.stats.stattools import durbin_watson
from statsmodels.stats.diagnostic import het_breuschpagan, het_white
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.api import OLS, add_constant
import statsmodels.api as sm

# Model Persistence
import pickle
import joblib

# Utilities
from datetime import datetime
import os

print("All libraries imported successfully!")
print(f"XGBoost Version: {XGBRegressor.__module__}")
print(f"Scikit-learn Version: {sm.__version__}")

All libraries imported successfully!
XGBoost Version: xgboost.sklearn
Scikit-learn Version: 0.14.6


## 2. Load and Explore Regression Dataset

### Dataset Information
We'll be using the **House Price Regression Dataset** which contains features related to housing characteristics and prices.

### Exploratory Data Analysis (EDA) Steps:
1. Load the dataset
2. Display basic information (shape, data types, memory usage)
3. Statistical summary (mean, std, min, max, quartiles)
4. Check for missing values
5. Visualize distributions
6. Analyze correlations

In [45]:
# Load the dataset
df = pd.read_csv('house_price_regression_dataset.csv')

# Display basic information
print("=" * 80)
print("DATASET OVERVIEW")
print("=" * 80)
print(f"\nDataset Shape: {df.shape[0]} rows × {df.shape[1]} columns")
print("\n" + "-" * 80)
print("Column Information:")
print("-" * 80)
print(df.info())
print("\n" + "=" * 80)

# Display first few rows
print("\nFirst 5 Rows:")
print("=" * 80)
df.head(10)

DATASET OVERVIEW

Dataset Shape: 12180 rows × 37 columns

--------------------------------------------------------------------------------
Column Information:
--------------------------------------------------------------------------------
<class 'pandas.DataFrame'>
RangeIndex: 12180 entries, 0 to 12179
Data columns (total 37 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   property_id                   12180 non-null  str    
 1   property_type                 12180 non-null  str    
 2   heating_type                  12180 non-null  str    
 3   parking_type                  12180 non-null  str    
 4   roof_type                     12180 non-null  str    
 5   has_pool                      12180 non-null  str    
 6   has_basement                  12180 non-null  str    
 7   hoa_flag                      12180 non-null  str    
 8   zipcode                       12180 non-null  str    
 9   builder_

,property_id,property_type,heating_type,parking_type,roof_type,has_pool,has_basement,hoa_flag,zipcode,builder_name,...,nearby_malls_count,annual_property_tax,hoa_fee,energy_efficiency_score,renovation_score,builder_reputation_score,random_hash_bucket,noise_feature_uniform,noise_feature_normal,sale_price
0,PROP_004768,Single_Family,Electric,Street,Tile,No,No,No,ZIP_083,Builder_010,...,1,5107.66,0.00,76.80,76.89,10.00,152,0.6983,-0.3285,2035398.89
1,PROP_008969,Condo,Electric,Covered,Asphalt,No,Yes,No,ZIP_080,Builder_061,...,5,5392.68,0.00,71.57,100.00,6.97,228,0.6528,-1.2570,1904372.82
2,PROP_001140,Apartment,Electric,Garage,Asphalt,No,No,Yes,ZIP_008,Builder_026,...,4,4477.96,125.19,58.42,62.98,9.05,9,0.6985,-0.6278,1190836.11
3,PROP_002373,Single_Family,Electric,Driveway,Flat,No,Yes,No,ZIP_055,Builder_058,...,3,4784.86,0.00,52.88,87.86,5.23,488,0.2902,-0.9624,1530678.30
4,PROP_007900,Condo,Electric,Driveway,Asphalt,No,No,No,ZIP_024,Builder_003,...,2,3567.30,0.00,64.55,78.00,10.00,467,0.0535,0.4386,1409852.98
5,PROP_002272,Single_Family,Heat_Pump,Garage,Metal,No,No,No,ZIP_084,Builder_011,...,3,3567.35,0.00,56.10,73.00,5.70,311,0.9205,0.9038,1124413.57
6,PROP_004990,Townhouse,Electric,Garage,Tile,Yes,No,No,ZIP_117,Builder_060,...,3,2448.00,0.00,42.61,86.77,8.53,401,0.2012,0.6842,1127532.08
7,PROP_008317,Single_Family,Electric,Driveway,Asphalt,No,Yes,Yes,ZIP_101,Builder_079,...,3,2958.32,84.32,60.00,82.86,6.99,107,0.8016,-0.9848,1174103.14
8,PROP_008848,Single_Family,Gas,Garage,Asphalt,No,No,No,ZIP_032,Builder_037,...,3,4908.91,0.00,62.89,78.09,8.30,421,0.3891,-0.4878,1443658.89
9,PROP_007766,Single_Family,Heat_Pump,Garage,Flat,No,No,Yes,ZIP_061,Builder_061,...,2,3355.26,140.47,49.79,79.67,6.29,37,0.9094,-0.2417,977339.26


In [46]:
# Statistical Summary
print("=" * 80)
print("STATISTICAL SUMMARY")
print("=" * 80)
print(df.describe())

print("\n" + "=" * 80)
print("MISSING VALUES")
print("=" * 80)
missing_data = pd.DataFrame({
    'Column': df.columns,
    'Missing_Count': df.isnull().sum(),
    'Missing_Percentage': (df.isnull().sum() / len(df)) * 100
})
missing_data = missing_data[missing_data['Missing_Count'] > 0].sort_values('Missing_Count', ascending=False)

if len(missing_data) > 0:
    print(missing_data.to_string(index=False))
else:
    print("✓ No missing values found in the dataset!")

STATISTICAL SUMMARY
          area_sqft  lot_size_sqft  num_bedrooms  num_bathrooms   garage_size  \
count  12180.000000   11415.000000  12180.000000   12180.000000  12180.000000   
mean    1743.031149    4051.702882      3.508128       3.199236      1.113465   
std      660.691917    2296.711315      1.308050       1.212275      0.696560   
min      350.000000     400.000000      1.000000       1.000000      0.000000   
25%     1339.600000    2444.250000      3.000000       2.300000      1.000000   
50%     1723.150000    3720.700000      3.000000       3.100000      1.000000   
75%     2086.425000    5285.550000      4.000000       4.000000      2.000000   
max     8520.700000   31620.000000      9.000000       7.000000      4.000000   

          house_age  distance_to_city_center_km  distance_to_metro_km  \
count  12180.000000                12180.000000          12180.000000   
mean      32.976125                   13.876133              5.009269   
std       17.436886            

### Distribution Visualization

Understanding the distribution of the target variable and features is crucial for:
- **Identifying skewness**: Helps decide if transformations are needed
- **Detecting outliers**: Extreme values that may affect model performance
- **Understanding data spread**: Variance and central tendency

In [47]:
# Identify target column (assuming last column or column with 'price' in name)
target_cols = [col for col in df.columns if 'price' in col.lower() or 'target' in col.lower()]
if target_cols:
    target_column = target_cols[0]
else:
    target_column = df.columns[-1]  # Use last column as default

print(f"Target Column: {target_column}")

# Visualize target distribution
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=(f'{target_column} Distribution (Histogram)', 
                   f'{target_column} Distribution (Box Plot)')
)

# Histogram
fig.add_trace(
    go.Histogram(x=df[target_column], name='Target', 
                nbinsx=50, marker_color='indianred'),
    row=1, col=1
)

# Box Plot
fig.add_trace(
    go.Box(y=df[target_column], name='Target', marker_color='lightseagreen'),
    row=1, col=2
)

fig.update_layout(
    height=400,
    showlegend=False,
    title_text=f"<b>Target Variable ({target_column}) Distribution Analysis</b>",
    title_x=0.5
)

fig.show()

# Statistics
print(f"\nTarget Variable Statistics:")
print(f"  Mean: {df[target_column].mean():.2f}")
print(f"  Median: {df[target_column].median():.2f}")
print(f"  Std Dev: {df[target_column].std():.2f}")
print(f"  Skewness: {df[target_column].skew():.2f}")
print(f"  Kurtosis: {df[target_column].kurtosis():.2f}")

Target Column: sale_price



Target Variable Statistics:
  Mean: 1200794.87
  Median: 1172469.69
  Std Dev: 381416.99
  Skewness: 0.99
  Kurtosis: 3.82


### Correlation Analysis

**Correlation** measures the strength and direction of linear relationships between variables:
- **Positive correlation**: Variables move in the same direction
- **Negative correlation**: Variables move in opposite directions
- **No correlation**: No linear relationship

**Correlation Coefficient (Pearson's r)** ranges from -1 to +1:
- r = +1: Perfect positive correlation
- r = -1: Perfect negative correlation
- r = 0: No linear correlation

In [48]:
# Calculate correlation matrix
correlation_matrix = df.corr(numeric_only = True)

# Create interactive heatmap
fig = go.Figure(data=go.Heatmap(
    z=correlation_matrix.values,
    x=correlation_matrix.columns,
    y=correlation_matrix.columns,
    colorscale='RdBu',
    zmid=0,
    text=correlation_matrix.values.round(2),
    texttemplate='%{text}',
    textfont={"size": 8},
    colorbar=dict(title="Correlation")
))

fig.update_layout(
    title='<b>Feature Correlation Heatmap</b>',
    title_x=0.5,
    width=900,
    height=700,
    xaxis_tickangle=-45
)

fig.show()

# Display correlations with target
print("\n" + "=" * 80)
print(f"CORRELATIONS WITH TARGET VARIABLE ({target_column})")
print("=" * 80)
target_corr = correlation_matrix[target_column].sort_values(ascending=False)
print(target_corr.to_string())


CORRELATIONS WITH TARGET VARIABLE (sale_price)
sale_price                      1.000000
annual_property_tax             0.834067
area_sqft                       0.588362
num_bedrooms                    0.475727
median_neighborhood_income_k    0.455307
school_rating                   0.447782
lot_size_sqft                   0.416959
num_bathrooms                   0.385326
energy_efficiency_score         0.127983
renovation_score                0.090042
garage_size                     0.062059
hoa_fee                         0.014671
nearby_hospitals_count          0.012397
noise_feature_normal            0.009033
nearby_malls_count              0.006080
random_hash_bucket             -0.001019
noise_feature_uniform          -0.001289
builder_reputation_score       -0.009436
house_age                      -0.094498
distance_to_metro_km           -0.096664
distance_to_city_center_km     -0.133181
crime_index                    -0.446063


## 3. Data Preprocessing and Feature Engineering

### Why Preprocessing is Important:
- **Different scales**: Features with larger scales can dominate the model
- **Missing values**: Can cause errors or bias in predictions
- **Categorical variables**: Need numerical encoding for algorithms
- **Outliers**: Extreme values can skew model performance

### Preprocessing Steps:
1. **Separate features and target**
2. **Handle missing values** (if any)
3. **Feature scaling**: Standardization (mean=0, std=1) or Normalization (range=[0,1])
4. **Note**: XGBoost doesn't require scaling, but Linear Regression benefits from it

In [ ]:
# df =df.drop(columns = ['property_id'])
# Invalid columns: property_type: str, heating_type: str, parking_type: str, roof_type: str, has_pool: str, 
# has_basement: str, hoa_flag: str, zipcode: str, builder_name: str, neighborhood: str, street_name: str, quality_grade: str, 
# condition_grade: str, school_zone_grade: str
# df['zipcode1'] = df['zipcode'].str[4:7]
# df =df.drop(columns = ['zipcode'])
# df = df.rename(columns={"zipcode1": "zipcode"})
# df['zipcode'] = df['zipcode'].astype(int)
df['school_zone_grade'].unique()

# df.head()
# heating_type ['Electric', 'Heat_Pump', 'Gas', 'Oil']
# parking_type ['Street', 'Covered', 'Garage', 'Driveway']
# roof_type ['Tile', 'Asphalt', 'Flat', 'Metal']
# has_pool ['No', 'Yes']
# has_basement ['No', 'Yes']
# hoa_flag ['No', 'Yes']
# quality_grade ['excellent', 'luxury', 'very_good', nan, 'good', 'poor', 'fair']
# condition_grade ['fair', 'excellent', 'very_good', 'good', 'needs_work']
# school_zone_grade ['A+', 'B', 'A', 'C']


<StringArray>
['A+', 'B', 'A', 'C']
Length: 4, dtype: str

In [10]:
X = df.drop(columns=[target_column])
y = df[target_column]

print("=" * 80)
print("DATA SPLITTING")
print("=" * 80)
print(f"Features (X) shape: {X.shape}")
print(f"Target (y) shape: {y.shape}")
print(f"\nFeature columns: {list(X.columns)}")

# Handle missing values if any
if X.isnull().sum().sum() > 0:
    print("\n⚠ Missing values detected. Filling with median for numeric and mode for categorical...")

    # Identify numeric and categorical columns
    numeric_cols = X.select_dtypes(include=np.number).columns
    categorical_cols = X.select_dtypes(include='object').columns

    # Fill numeric columns with median
    if len(numeric_cols) > 0:
        X[numeric_cols] = X[numeric_cols].fillna(X[numeric_cols].median())

    # Fill categorical columns with mode (handle potential multiple modes by taking the first)
    if len(categorical_cols) > 0:
        for col in categorical_cols:
            if X[col].isnull().any():
                mode_val = X[col].mode()[0] # .mode() returns a Series, take the first mode
                X[col] = X[col].fillna(mode_val)

if y.isnull().sum() > 0:
    print("⚠ Missing values in target. Filling with median...")
    y = y.fillna(y.median())

# Create scaler for Linear Regression (we'll use later)
scaler = StandardScaler()

print("\n✓ Preprocessing setup complete!")
print("\nNote: We'll scale data separately for each train-test split to avoid data leakage.")

DATA SPLITTING
Features (X) shape: (12180, 36)
Target (y) shape: (12180,)

Feature columns: ['property_id', 'property_type', 'heating_type', 'parking_type', 'roof_type', 'has_pool', 'has_basement', 'hoa_flag', 'zipcode', 'builder_name', 'neighborhood', 'street_name', 'quality_grade', 'condition_grade', 'school_zone_grade', 'area_sqft', 'lot_size_sqft', 'num_bedrooms', 'num_bathrooms', 'garage_size', 'house_age', 'distance_to_city_center_km', 'distance_to_metro_km', 'school_rating', 'crime_index', 'median_neighborhood_income_k', 'nearby_hospitals_count', 'nearby_malls_count', 'annual_property_tax', 'hoa_fee', 'energy_efficiency_score', 'renovation_score', 'builder_reputation_score', 'random_hash_bucket', 'noise_feature_uniform', 'noise_feature_normal']

⚠ Missing values detected. Filling with median for numeric and mode for categorical...

✓ Preprocessing setup complete!

Note: We'll scale data separately for each train-test split to avoid data leakage.


## 4. Train-Test Split Strategies

### Why Multiple Split Strategies?
Different split ratios affect model performance and generalization:

**Split Ratio Selection Criteria:**
- **70-30 split**: Balanced approach, suitable for medium datasets (10K-100K samples)
- **80-20 split**: Most common split, preferred when you have enough data (>10K samples)
- **90-10 split**: Used when you have very large datasets (>100K samples) or small test sets are acceptable

### Data Leakage Prevention:
Always split BEFORE scaling to prevent information from test set influencing training set.

### Cross-Validation vs Single Split:
- **Single split**: Fast, but results depend on the random split
- **K-Fold CV**: More robust, averages performance across K different splits

In [11]:
# Create multiple train-test splits
splits = {}

# Split 1: 70-30
X_train_70, X_test_70, y_train_70, y_test_70 = train_test_split(
    X, y, test_size=0.30, random_state=42
)
splits['70-30'] = (X_train_70, X_test_70, y_train_70, y_test_70)

# Split 2: 80-20 (Primary split we'll use)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)
splits['80-20'] = (X_train, X_test, y_train, y_test)

# Split 3: 90-10
X_train_90, X_test_90, y_train_90, y_test_90 = train_test_split(
    X, y, test_size=0.10, random_state=42
)
splits['90-10'] = (X_train_90, X_test_90, y_train_90, y_test_90)

# Display split information
print("=" * 80)
print("TRAIN-TEST SPLIT SUMMARY")
print("=" * 80)
for split_name, (X_tr, X_te, y_tr, y_te) in splits.items():
    print(f"\n{split_name} Split:")
    print(f"  Training samples: {len(X_tr):,} ({len(X_tr)/len(X)*100:.1f}%)")
    print(f"  Testing samples:  {len(X_te):,} ({len(X_te)/len(X)*100:.1f}%)")
    
print("\n" + "=" * 80)
print("PRIMARY SPLIT: 80-20 (balanced approach)")
print("=" * 80)
print(f"✓ X_train shape: {X_train.shape}")
print(f"✓ X_test shape:  {X_test.shape}")
print(f"✓ y_train shape: {y_train.shape}")
print(f"✓ y_test shape:  {y_test.shape}")

TRAIN-TEST SPLIT SUMMARY

70-30 Split:
  Training samples: 8,526 (70.0%)
  Testing samples:  3,654 (30.0%)

80-20 Split:
  Training samples: 9,744 (80.0%)
  Testing samples:  2,436 (20.0%)

90-10 Split:
  Training samples: 10,962 (90.0%)
  Testing samples:  1,218 (10.0%)

PRIMARY SPLIT: 80-20 (balanced approach)
✓ X_train shape: (9744, 36)
✓ X_test shape:  (2436, 36)
✓ y_train shape: (9744,)
✓ y_test shape:  (2436,)


---
#  PART 1: XGBoost Regressor Analysis
---

## 5. XGBoost Regressor - Introduction

### What is XGBoost?
**XGBoost (eXtreme Gradient Boosting)** is an advanced implementation of gradient boosting algorithm:
- **Gradient Boosting**: Ensemble method that builds trees sequentially, each correcting errors of previous trees
- **Boosting**: Combines weak learners (shallow trees) into a strong learner
- **Optimization**: Uses gradient descent to minimize loss function

### How XGBoost Works:
1. **Start with initial prediction** (mean of target)
2. **Calculate residuals** (errors) from current prediction
3. **Build a tree** to predict these residuals
4. **Update predictions** by adding tree's output (scaled by learning rate)
5. **Repeat** until number of trees (n_estimators) is reached

### Key Hyperparameters Explained:

**Tree Structure:**
- `n_estimators`: Number of trees (more trees = more complex model, but can overfit)
- `max_depth`: Maximum tree depth (deeper = more complex interactions, but can overfit)
- `min_child_weight`: Minimum sum of instance weight in a child (controls overfitting)

**Learning Process:**
- `learning_rate` (eta): Step size shrinkage (0.01-0.3, smaller = more conservative updates)
- `subsample`: Fraction of samples used per tree (0.5-1.0, prevents overfitting)
- `colsample_bytree`: Fraction of features used per tree (0.5-1.0, adds randomness)

**Regularization:**
- `gamma`: Minimum loss reduction to split a node (higher = more conservative)
- `reg_alpha` (L1): Adds penalty on leaf weights magnitude (feature selection)
- `reg_lambda` (L2): Adds penalty on leaf weights squared (smooths weights)

### Advantages:
✓ Handles missing values automatically
✓ Does NOT require feature scaling
✓ Fast training with parallel processing
✓ Built-in regularization prevents overfitting
✓ Excellent for structured/tabular data

### When to Use XGBoost:
- Structured tabular data with complex non-linear relationships
- When model interpretability is less important than accuracy
- When you have sufficient data (1000+ samples)
- Competition-level performance needed

### 5.1 Baseline XGBoost Model

Start with default parameters to establish a performance baseline before optimization.

In [ ]:
# Initialize baseline XGBoost regressor
xgb_baseline = XGBRegressor(
    random_state=42,
    n_jobs=-1  # Use all CPU cores
)

# Train the model
print("Training baseline XGBoost model...")
xgb_baseline.fit(X_train, y_train)

# Output is truncated. View as a scrollable element or open in a text editor. Adjust cell output settings...
# Make predictions
y_pred_baseline = xgb_baseline.predict(X_test)

# Evaluation metrics
mse_baseline = mean_squared_error(y_test, y_pred_baseline)
rmse_baseline = np.sqrt(mse_baseline)
mae_baseline = mean_absolute_error(y_test, y_pred_baseline)
r2_baseline = r2_score(y_test, y_pred_baseline)

print("\n" + "=" * 80)
print("BASELINE XGBoost PERFORMANCE")
print("=" * 80)
print(f"Mean Squared Error (MSE):       {mse_baseline:,.2f}")
print(f"Root Mean Squared Error (RMSE): {rmse_baseline:,.2f}")
print(f"Mean Absolute Error (MAE):      {mae_baseline:,.2f}")
print(f"R² Score:                       {r2_baseline:.4f}")
print("\nInterpretation:")
print(f"  - Model explains {r2_baseline*100:.2f}% of variance in target")
print(f"  - Average prediction error: {mae_baseline:,.2f}")
print("=" * 80)

Training baseline XGBoost model...


ValueError: DataFrame.dtypes for data must be int, float, bool or category. When categorical type is supplied, the experimental DMatrix parameter`enable_categorical` must be set to `True`.  Invalid columns:property_id: str, property_type: str, heating_type: str, parking_type: str, roof_type: str, has_pool: str, has_basement: str, hoa_flag: str, zipcode: str, builder_name: str, neighborhood: str, street_name: str, quality_grade: str, condition_grade: str, school_zone_grade: str

### 5.2 XGBoost - Grid Search Hyperparameter Tuning

**Grid Search** is an exhaustive search through a manually specified hyperparameter space.

#### How Grid Search Works:
1. Define a grid of hyperparameter combinations
2. Train a model for EVERY combination
3. Use cross-validation to evaluate each
4. Select the combination with best performance

#### Advantages:
✓ Guaranteed to find the best combination in the defined grid
✓ Comprehensive exploration
✓ Reproducible results

#### Disadvantages:
✗ **Computationally expensive**: O(n^p) where n=values per parameter, p=number of parameters
✗ Exponential growth with more parameters
✗ May miss optimal values between grid points

#### When to Use:
- Small hyperparameter space (2-3 parameters with 3-5 values each)
- When computational resources are available
- For final tuning after narrowing Search space

#### Cross-Validation Strategy:
We use **5-fold CV**: Data is split into 5 parts, model trained on 4 and validated on 1, repeated 5 times. Final score is the average.

#### Grid Search Complexity:
If we search 3 values for 4 parameters = 3^4 = 81 combinations × 5 folds = **405 model trainings!**

In [14]:
# Define parameter grid for Grid Search
param_grid_xgb = {
    'n_estimators': [100, 200, 300],          # Number of boosting rounds
    'max_depth': [3, 5, 7],                   # Maximum tree depth
    'learning_rate': [0.01, 0.1, 0.2],        # Step size shrinkage
    'subsample': [0.8, 1.0],                  # Subsample ratio of training instances
    'colsample_bytree': [0.8, 1.0]            # Subsample ratio of features
}

# Calculate total combinations
total_combinations = np.prod([len(v) for v in param_grid_xgb.values()])
print("=" * 80)
print("GRID SEARCH CONFIGURATION")
print("=" * 80)
print(f"Parameter grid: {param_grid_xgb}")
print(f"\nTotal combinations: {total_combinations}")
print(f"With 5-fold CV: {total_combinations * 5} model trainings")
print("=" * 80)

# Initialize GridSearchCV
grid_search_xgb = GridSearchCV(
    estimator=XGBRegressor(random_state=42, n_jobs=-1),
    param_grid=param_grid_xgb,
    scoring='neg_mean_squared_error',  # Negative MSE (higher is better)
    cv=5,                              # 5-fold cross-validation
    verbose=2,                         # Show progress
    n_jobs=-1                          # Parallel processing
)

# Perform Grid Search
print("\n🔍 Starting Grid Search...")
print("This may take several minutes depending on dataset size...")
start_time = datetime.now()

grid_search_xgb.fit(X_train, y_train)

end_time = datetime.now()
duration = (end_time - start_time).total_seconds()

# Display results
print("\n" + "=" * 80)
print("GRID SEARCH RESULTS")
print("=" * 80)
print(f"✓ Search completed in {duration:.1f} seconds ({duration/60:.1f} minutes)")
print(f"\nBest Parameters:")
for param, value in grid_search_xgb.best_params_.items():
    print(f"  {param}: {value}")

print(f"\nBest CV Score (Negative MSE): {grid_search_xgb.best_score_:.2f}")
print(f"Best CV RMSE: {np.sqrt(-grid_search_xgb.best_score_):,.2f}")

# Test set evaluation
y_pred_grid = grid_search_xgb.best_estimator_.predict(X_test)
mse_grid = mean_squared_error(y_test, y_pred_grid)
rmse_grid = np.sqrt(mse_grid)
mae_grid = mean_absolute_error(y_test, y_pred_grid)
r2_grid = r2_score(y_test, y_pred_grid)

print(f"\nTest Set Performance:")
print(f"  RMSE: {rmse_grid:,.2f}")
print(f"  MAE:  {mae_grid:,.2f}")
print(f"  R²:   {r2_grid:.4f}")

print(f"\nImprovement over baseline:")
print(f"  RMSE: {((rmse_baseline - rmse_grid)/rmse_baseline)*100:+.2f}%")
print(f"  R²:   {((r2_grid - r2_baseline)/r2_baseline)*100:+.2f}%")
print("=" * 80)

GRID SEARCH CONFIGURATION
Parameter grid: {'n_estimators': [100, 200, 300], 'max_depth': [3, 5, 7], 'learning_rate': [0.01, 0.1, 0.2], 'subsample': [0.8, 1.0], 'colsample_bytree': [0.8, 1.0]}

Total combinations: 108
With 5-fold CV: 540 model trainings

🔍 Starting Grid Search...
This may take several minutes depending on dataset size...
Fitting 5 folds for each of 108 candidates, totalling 540 fits


ValueError: 
All the 540 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
540 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Users\Malhar Jojare\Desktop\Technical Training\env\Lib\site-packages\xgboost\data.py", line 408, in pandas_feature_info
    new_feature_types.append(_pandas_dtype_mapper[dtype.name])
                             ~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^
KeyError: 'str'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "c:\Users\Malhar Jojare\Desktop\Technical Training\env\Lib\site-packages\sklearn\model_selection\_validation.py", line 851, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Ma

### 5.3 XGBoost - Random Search Hyperparameter Tuning

**Random Search** samples random combinations from hyperparameter distributions.

#### How Random Search Works:
1. Define distributions for each hyperparameter (not discrete grids)
2. Randomly sample n_iter combinations
3. Train and evaluate each using cross-validation
4. Select the best performed combination

#### Advantages:
✓ **More efficient** than Grid Search for large spaces
✓ Can explore continuous distributions
✓ Often finds better parameters with fewer iterations
✓ Better exploration of parameter space
✓ Time budget easily controlled with n_iter

#### Disadvantages:
✗ No guarantee of finding optimal combination
✗ Results vary between runs (unless random_state is set)
✗ May miss important regions

#### When to Use:
- Large hyperparameter space (>4 parameters)
- When you want to explore broader distributions
- Limited computational time
- Initial exploration before Grid Search

#### Scientific Evidence:
Research (Bergstra & Bengio, 2012) shows random search often outperforms grid search with the same budget!

#### Key Insight:
Not all hyperparameters equally important. Random search is better at finding important ones!

In [15]:
# Define parameter distributions for Random Search
param_dist_xgb = {
    'n_estimators': [50, 100, 150, 200, 250, 300, 350],
    'max_depth': [3, 4, 5, 6, 7, 8, 9],
    'learning_rate': [0.01, 0.05, 0.1, 0.15, 0.2, 0.25, 0.3],
    'subsample': [0.6, 0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0],
    'gamma': [0, 0.1, 0.2, 0.3, 0.4],
    'min_child_weight': [1, 3, 5, 7]
}

# Calculate possible combinations
possible_combinations = np.prod([len(v) for v in param_dist_xgb.values()])
n_iter = 50  # Number of random samples

print("=" * 80)
print("RANDOM SEARCH CONFIGURATION")
print("=" * 80)
print(f"Possible combinations: {possible_combinations:,}")
print(f"Random samples (n_iter): {n_iter}")
print(f"Exploration rate: {(n_iter/possible_combinations)*100:.2f}%")
print(f"With 5-fold CV: {n_iter * 5} model trainings (vs {total_combinations * 5} for full grid)")
print("=" * 80)

# Initialize RandomizedSearchCV
random_search_xgb = RandomizedSearchCV(
    estimator=XGBRegressor(random_state=42, n_jobs=-1),
    param_distributions=param_dist_xgb,
    n_iter=n_iter,                     # Number of random combinations to try
    scoring='neg_mean_squared_error',
    cv=5,
    verbose=2,
    random_state=42,                   # For reproducibility
    n_jobs=-1
)

# Perform Random Search
print("\n🎲 Starting Random Search...")
start_time = datetime.now()

random_search_xgb.fit(X_train, y_train)

end_time = datetime.now()
duration = (end_time - start_time).total_seconds()

# Display results
print("\n" + "=" * 80)
print("RANDOM SEARCH RESULTS")
print("=" * 80)
print(f"✓ Search completed in {duration:.1f} seconds ({duration/60:.1f} minutes)")
print(f"\nBest Parameters:")
for param, value in random_search_xgb.best_params_.items():
    print(f"  {param}: {value}")

print(f"\nBest CV Score (Negative MSE): {random_search_xgb.best_score_:.2f}")
print(f"Best CV RMSE: {np.sqrt(-random_search_xgb.best_score_):,.2f}")

# Test set evaluation
y_pred_random = random_search_xgb.best_estimator_.predict(X_test)
mse_random = mean_squared_error(y_test, y_pred_random)
rmse_random = np.sqrt(mse_random)
mae_random = mean_absolute_error(y_test, y_pred_random)
r2_random = r2_score(y_test, y_pred_random)

print(f"\nTest Set Performance:")
print(f"  RMSE: {rmse_random:,.2f}")
print(f"  MAE:  {mae_random:,.2f}")
print(f"  R²:   {r2_random:.4f}")

print(f"\nComparison with Grid Search:")
print(f"  RMSE difference: {rmse_random - rmse_grid:+.2f}")
print(f"  R² difference:   {r2_random - r2_grid:+.4f}")
print(f"  Time saved: {((duration)/duration)*100 if duration > 0 else 0:.1f}%")
print("=" * 80)

RANDOM SEARCH CONFIGURATION
Possible combinations: 171,500
Random samples (n_iter): 50
Exploration rate: 0.03%
With 5-fold CV: 250 model trainings (vs 540 for full grid)

🎲 Starting Random Search...
Fitting 5 folds for each of 50 candidates, totalling 250 fits


ValueError: 
All the 250 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
250 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Users\Malhar Jojare\Desktop\Technical Training\env\Lib\site-packages\xgboost\data.py", line 408, in pandas_feature_info
    new_feature_types.append(_pandas_dtype_mapper[dtype.name])
                             ~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^
KeyError: 'str'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "c:\Users\Malhar Jojare\Desktop\Technical Training\env\Lib\site-packages\sklearn\model_selection\_validation.py", line 851, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Ma

### 5.4 XGBoost - Bayesian Optimization Hyperparameter Tuning

**Bayesian Optimization** is an intelligent, sequential model-based optimization strategy.

#### How Bayesian Optimization Works:
1. **Build a surrogate model** (Gaussian Process) of the objective function
2. Use the model to **estimate performance** for unexplored regions
3. **Acquisition function** determines next point to try (balancing exploration/exploitation)
4. **Update surrogate model** with new results
5. **Iterate** until budget exhausted

#### Key Concepts:

**Surrogate Model (Gaussian Process):**
- Probabilistic model that approximates the (unknown) objective function
- Provides both **mean** (expected performance) and **uncertainty** estimates
- Updated with each new evaluation

**Acquisition Function:**
- **Expected Improvement (EI)**: How much better than current best?
- **Upper Confidence Bound (UCB)**: Balance mean + uncertainty
- **Probability of Improvement (PI)**: Chance of beating current best

**Exploration vs Exploitation:**
- **Exploitation**: Search near known good parameters
- **Exploration**: Try uncertain regions that might be better
- Acquisition function balances both!

#### Advantages:
✓ **Most efficient**: Finds good parameters with fewest evaluations
✓ **Intelligent search**: Learns from previous evaluations
✓ **Handles expensive functions**: Ideal when training is slow
✓ **Continuous space**: Can sample any value in range
✓ **Quantifies uncertainty**: Knows where it's uncertain

#### Disadvantages:
✗ More complex to implement and understand
✗ Overhead in building surrogate model
✗ May get stuck in local optima
✗ Requires more sophisticated libraries (scikit-optimize, Optuna, Hyperopt)

#### When to Use:
- **Limited budget**: Can only afford 20-50 evaluations
- **Expensive models**: Each training takes minutes/hours
- **Continuous hyperparameters**: Better than discrete grids
- **Production projects**: Want best possible performance

#### Mathematical Foundation:
The acquisition function α(x) guides the search:
- **EI(x) = E[max(f(x) - f(x⁺), 0)]** where x⁺ is current best
- Higher EI → more promising region

In [16]:
# Define search space for Bayesian Optimization using scikit-optimize
search_space_xgb = {
    'n_estimators': Integer(50, 350, name='n_estimators'),
    'max_depth': Integer(3, 10, name='max_depth'),
    'learning_rate': Real(0.01, 0.3, prior='log-uniform', name='learning_rate'),
    'subsample': Real(0.6, 1.0, name='subsample'),
    'colsample_bytree': Real(0.6, 1.0, name='colsample_bytree'),
    'gamma': Real(0, 0.5, name='gamma'),
    'min_child_weight': Integer(1, 10, name='min_child_weight'),
    'reg_alpha': Real(0, 1.0, name='reg_alpha'),            # L1 regularization
    'reg_lambda': Real(0, 1.0, name='reg_lambda')           # L2 regularization
}

n_iter_bayes = 30  # Number of Bayesian optimization iterations

print("=" * 80)
print("BAYESIAN OPTIMIZATION CONFIGURATION")
print("=" * 80)
print("Search Space:")
for param, space in search_space_xgb.items():
    print(f"  {param}: {space}")
print(f"\nNumber of iterations: {n_iter_bayes}")
print(f"With 5-fold CV: {n_iter_bayes * 5} model trainings")
print("\n⚙ Strategy:")
print("  - Using Gaussian Process as surrogate model")
print("  - Expected Improvement as acquisition function")
print("  - Balancing exploration and exploitation")
print("=" * 80)

# Initialize BayesSearchCV
bayes_search_xgb = BayesSearchCV(
    estimator=XGBRegressor(random_state=42, n_jobs=-1),
    search_spaces=search_space_xgb,
    n_iter=n_iter_bayes,               # Number of optimization iterations
    scoring='neg_mean_squared_error',
    cv=5,
    verbose=2,
    random_state=42,
    n_jobs=-1
)

# Perform Bayesian Optimization
print("\n🧠 Starting Bayesian Optimization...")
print("Intelligent sequential search in progress...")
start_time = datetime.now()

bayes_search_xgb.fit(X_train, y_train)

end_time = datetime.now()
duration = (end_time - start_time).total_seconds()

# Display results
print("\n" + "=" * 80)
print("BAYESIAN OPTIMIZATION RESULTS")
print("=" * 80)
print(f"✓ Optimization completed in {duration:.1f} seconds ({duration/60:.1f} minutes)")
print(f"\nBest Parameters Found:")
for param, value in bayes_search_xgb.best_params_.items():
    print(f"  {param}: {value}")

print(f"\nBest CV Score (Negative MSE): {bayes_search_xgb.best_score_:.2f}")
print(f"Best CV RMSE: {np.sqrt(-bayes_search_xgb.best_score_):,.2f}")

# Test set evaluation
y_pred_bayes = bayes_search_xgb.best_estimator_.predict(X_test)
mse_bayes = mean_squared_error(y_test, y_pred_bayes)
rmse_bayes = np.sqrt(mse_bayes)
mae_bayes = mean_absolute_error(y_test, y_pred_bayes)
r2_bayes = r2_score(y_test, y_pred_bayes)

print(f"\nTest Set Performance:")
print(f"  RMSE: {rmse_bayes:,.2f}")
print(f"  MAE:  {mae_bayes:,.2f}")
print(f"  R²:   {r2_bayes:.4f}")

print(f"\nComparison with Other Methods:")
print(f"  vs Baseline  - RMSE: {rmse_baseline - rmse_bayes:+.2f}, R²: {r2_bayes - r2_baseline:+.4f}")
print(f"  vs Grid      - RMSE: {rmse_grid - rmse_bayes:+.2f}, R²: {r2_bayes - r2_grid:+.4f}")
print(f"  vs Random    - RMSE: {rmse_random - rmse_bayes:+.2f}, R²: {r2_bayes - r2_random:+.4f}")
print("=" * 80)

# Store the best XGBoost model
xgb_best_model = bayes_search_xgb.best_estimator_
print(f"\n✓ Best XGBoost model saved as 'xgb_best_model'")

BAYESIAN OPTIMIZATION CONFIGURATION
Search Space:
  n_estimators: Integer(low=50, high=350, prior='uniform', transform='identity')
  max_depth: Integer(low=3, high=10, prior='uniform', transform='identity')
  learning_rate: Real(low=0.01, high=0.3, prior='log-uniform', transform='identity')
  subsample: Real(low=0.6, high=1.0, prior='uniform', transform='identity')
  colsample_bytree: Real(low=0.6, high=1.0, prior='uniform', transform='identity')
  gamma: Real(low=0, high=0.5, prior='uniform', transform='identity')
  min_child_weight: Integer(low=1, high=10, prior='uniform', transform='identity')
  reg_alpha: Real(low=0, high=1.0, prior='uniform', transform='identity')
  reg_lambda: Real(low=0, high=1.0, prior='uniform', transform='identity')

Number of iterations: 30
With 5-fold CV: 150 model trainings

⚙ Strategy:
  - Using Gaussian Process as surrogate model
  - Expected Improvement as acquisition function
  - Balancing exploration and exploitation

🧠 Starting Bayesian Optimization.

ValueError: DataFrame.dtypes for data must be int, float, bool or category. When categorical type is supplied, the experimental DMatrix parameter`enable_categorical` must be set to `True`.  Invalid columns:property_id: str, property_type: str, heating_type: str, parking_type: str, roof_type: str, has_pool: str, has_basement: str, hoa_flag: str, zipcode: str, builder_name: str, neighborhood: str, street_name: str, quality_grade: str, condition_grade: str, school_zone_grade: str

### 5.5 XGBoost - Model Evaluation and Visualization

Comprehensive visualization helps understand model performance and identify issues.

In [ ]:
# Create comprehensive visualizations for XGBoost

# 1. Actual vs Predicted Plot
fig1 = go.Figure()

# Add scatter plot
fig1.add_trace(go.Scatter(
    x=y_test,
    y=y_pred_bayes,
    mode='markers',
    name='Predictions',
    marker=dict(size=8, color='royalblue', opacity=0.6),
    text=[f'Actual: {a:.2f}<br>Predicted: {p:.2f}<br>Error: {a-p:.2f}' 
          for a, p in zip(y_test, y_pred_bayes)],
    hovertemplate='%{text}<extra></extra>'
))

# Add perfect prediction line
min_val = min(y_test.min(), y_pred_bayes.min())
max_val = max(y_test.max(), y_pred_bayes.max())
fig1.add_trace(go.Scatter(
    x=[min_val, max_val],
    y=[min_val, max_val],
    mode='lines',
    name='Perfect Prediction',
    line=dict(color='red', dash='dash', width=2)
))

fig1.update_layout(
    title=f'<b>XGBoost: Actual vs Predicted Values</b><br><sub>R² = {r2_bayes:.4f}, RMSE = {rmse_bayes:,.2f}</sub>',
    xaxis_title='Actual Values',
    yaxis_title='Predicted Values',
    width=800,
    height=600,
    hovermode='closest',
    showlegend=True
)

fig1.show()

# 2. Residual Plot
residuals = y_test - y_pred_bayes

fig2 = go.Figure()

fig2.add_trace(go.Scatter(
    x=y_pred_bayes,
    y=residuals,
    mode='markers',
    marker=dict(size=8, color=residuals, colorscale='RdYlBu', 
                showscale=True, colorbar=dict(title="Residual")),
    text=[f'Predicted: {p:.2f}<br>Residual: {r:.2f}' 
          for p, r in zip(y_pred_bayes, residuals)],
    hovertemplate='%{text}<extra></extra>'
))

# Add zero line
fig2.add_hline(y=0, line_dash="dash", line_color="red", annotation_text="Zero Error")

# Add ±2σ lines
std_residuals = residuals.std()
fig2.add_hline(y=2*std_residuals, line_dash="dot", line_color="orange", 
               annotation_text="+2σ", annotation_position="right")
fig2.add_hline(y=-2*std_residuals, line_dash="dot", line_color="orange", 
               annotation_text="-2σ", annotation_position="right")

fig2.update_layout(
    title='<b>XGBoost: Residual Plot</b><br><sub>Residuals should be randomly scattered around zero</sub>',
    xaxis_title='Predicted Values',
    yaxis_title='Residuals (Actual - Predicted)',
    width=800,
    height=600,
    hovermode='closest'
)

fig2.show()

print("📊 Residual Analysis:")
print(f"  Mean residual: {residuals.mean():.4f} (should be ≈ 0)")
print(f"  Std residual:  {residuals.std():.4f}")
print(f"  Min residual:  {residuals.min():.2f}")
print(f"  Max residual:  {residuals.max():.2f}")

In [ ]:
# 3. Feature Importance
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': xgb_best_model.feature_importances_
}).sort_values('importance', ascending=True)

fig3 = go.Figure(go.Bar(
    x=feature_importance['importance'],
    y=feature_importance['feature'],
    orientation='h',
    marker=dict(
        color=feature_importance['importance'],
        colorscale='Viridis',
        showscale=True,
        colorbar=dict(title="Importance")
    ),
    text=feature_importance['importance'].round(4),
    textposition='outside'
))

fig3.update_layout(
    title='<b>XGBoost: Feature Importance</b><br><sub>Higher values indicate more important features</sub>',
    xaxis_title='Importance Score',
    yaxis_title='Features',
    width=800,
    height=max(400, len(X.columns) * 25),
    showlegend=False
)

fig3.show()

print("\n📈 Top 5 Most Important Features:")
top_features = feature_importance.tail(5)
for idx, row in top_features.iterrows():
    print(f"  {row['feature']}: {row['importance']:.4f}")

### 5.6 XGBoost - Model Persistence (Pickle and Joblib)

**Model Persistence** allows saving trained models to disk for later use without retraining.

#### Pickle vs Joblib:

**Pickle (Python Standard Library):**
- ✓ Built into Python, no extra dependencies
- ✓ Can serialize any Python object
- ✗ Slower for large numpy arrays
- ✗ Larger file sizes
- **Use case**: Small models, general Python objects

**Joblib (from sklearn.external):**
- ✓ Optimized for numpy arrays (3-10x faster)
- ✓ Smaller file sizes with compression
- ✓ More efficient for scikit-learn models
- ✗ Requires joblib library
- **Use case**: ML models, large datasets

#### Best Practices:
1. Save model metadata (hyperparameters, metrics, feature names)
2. Version models (model_v1.pkl, model_v2.pkl)
3. Document training date and data version
4. Test loaded model predictions match original

In [ ]:
# Create a models directory
os.makedirs('models', exist_ok=True)

# Prepare model metadata
xgb_metadata = {
    'model_name': 'XGBoost Regressor',
    'created_date': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'hyperparameters': xgb_best_model.get_params(),
    'feature_names': list(X.columns),
    'metrics': {
        'RMSE': rmse_bayes,
        'MAE': mae_bayes,
        'R2': r2_bayes
    },
    'tuning_method': 'Bayesian Optimization',
    'train_size': len(X_train),
    'test_size': len(X_test)
}

print("=" * 80)
print("SAVING XGBOOST MODEL")
print("=" * 80)

# 1. Save using Pickle
pickle_filename = 'models/xgboost_model.pkl'
with open(pickle_filename, 'wb') as file:
    pickle.dump(xgb_best_model, file)
print(f"✓ Model saved with pickle: {pickle_filename}")
print(f"  File size: {os.path.getsize(pickle_filename) / 1024:.2f} KB")

# 2. Save using Joblib
joblib_filename = 'models/xgboost_model.joblib'
joblib.dump(xgb_best_model, joblib_filename, compress=3)  # compression level 3
print(f"✓ Model saved with joblib: {joblib_filename}")
print(f"  File size: {os.path.getsize(joblib_filename) / 1024:.2f} KB")

# 3. Save metadata
metadata_filename = 'models/xgboost_metadata.pkl'
with open(metadata_filename, 'wb') as file:
    pickle.dump(xgb_metadata, file)
print(f"✓ Metadata saved: {metadata_filename}")

print("\n" + "=" * 80)
print("LOADING AND TESTING MODELS")
print("=" * 80)

# Load using Pickle
with open(pickle_filename, 'rb') as file:
    xgb_loaded_pickle = pickle.load(file)
y_pred_pickle = xgb_loaded_pickle.predict(X_test[:5])
print(f"✓ Model loaded from pickle successfully")

# Load using Joblib
xgb_loaded_joblib = joblib.load(joblib_filename)
y_pred_joblib = xgb_loaded_joblib.predict(X_test[:5])
print(f"✓ Model loaded from joblib successfully")

# Verify predictions match
print(f"\n🔍 Verification (first 5 predictions):")
print(f"Original:     {y_pred_bayes[:5]}")
print(f"From Pickle:  {y_pred_pickle}")
print(f"From Joblib:  {y_pred_joblib}")

if np.allclose(y_pred_bayes[:5], y_pred_pickle) and np.allclose(y_pred_bayes[:5], y_pred_joblib):
    print("\n✅ SUCCESS: All models produce identical predictions!")
else:
    print("\n⚠ WARNING: Predictions differ!")

print("=" * 80)

---
# 📐 PART 2: Linear Regression Analysis
---

## 6. Linear Regression - Introduction

### What is Linear Regression?
**Linear Regression** is a fundamental statistical method for modeling the relationship between a dependent variable and one or more independent variables.

### Mathematical  Formula:
For multiple linear regression:
$$y = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + ... + \beta_n x_n + \epsilon$$

Where:
- $y$ = Target variable (dependent variable)
- $x_1, x_2, ..., x_n$ = Features (independent variables)
- $\beta_0$ = Intercept (y-value when all x = 0)
- $\beta_1, ..., \beta_n$ = Coefficients (slopes, effect of each feature)
- $\epsilon$ = Error term (residuals)

### How Linear Regression Works:

**1. Ordinary Least Squares (OLS) Method:**
Finds coefficients that minimize the sum of squared residuals:
$$\text{minimize} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2$$

**2. Closed-Form Solution (Normal Equation):**
$$\beta = (X^T X)^{-1} X^T y$$

**3. Gradient Descent Method:**
Iteratively updates coefficients:
$$\beta_{new} = \beta_{old} - \alpha \frac{\partial J}{\partial \beta}$$

### Key Assumptions (MUST BE CHECKED!):
1. **Linearity**: Relationship between X and y is linear
2. **Independence**: Observations are independent
3. **Homoscedasticity**: Constant variance of residuals
4. **Normality**: Residuals are normally distributed
5. **No Multicollinearity**: Features are not highly correlated

### Advantages:
✓ **Interpretable**: Clear coefficient interpretation
✓ **Fast**: Closed-form solution (no iterative training)
✓ **Statistical inference**: Confidence intervals, p-values
✓ **Low overfitting**: Simplicity prevents overfitting on small data
✓ **Baseline**: Excellent starting point for regression

### Disadvantages:
✗ **Linear relationships only**: Cannot capture non-linear patterns
✗ **Sensitive to outliers**: Squared error magnifies large residuals
✗ **Multicollinearity issues**: Unstable coefficients with correlated features
✗ **Assumptions required**: Violations affect validity

### When to Use:
- Linear relationships between variables
- Need coefficient interpretation
- Small to medium datasets
- Statistical hypothesis testing required
- Baseline model for comparison

### 6.1 Data Scaling for Linear Regression

**Why scaling is crucial for Linear Regression:**
- Features on different scales can dominate the cost function
- Gradient descent converges faster with scaled features
- Regularization (Ridge, Lasso) requires features on similar scales
- Coefficient interpretation becomes easier

**StandardScaler** transforms features to have:
- Mean (μ) = 0
- Standard Deviation (σ) = 1
- Formula: $z = \frac{x - \mu}{\sigma}$

In [ ]:
# Scale the data for Linear Regression
scaler = StandardScaler()

# Fit scaler on training data only (prevent data leakage!)
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("=" * 80)
print("DATA SCALING")
print("=" * 80)
print("Before Scaling (Training Set):")
print(f"  Mean: {X_train.mean().mean():.4f}")
print(f"  Std:  {X_train.std().mean():.4f}")

print("\nAfter Scaling (Training Set):")
print(f"  Mean: {X_train_scaled.mean():.4f} (should be ≈ 0)")
print(f"  Std:  {X_train_scaled.std():.4f} (should be ≈ 1)")

print("\n✓ Scaling completed!")
print("  - Scaler fitted on training data")
print("  - Transformation applied to both train and test sets")
print("=" * 80)

### 6.2 Baseline Linear Regression Model

In [ ]:
# Initialize and train baseline Linear Regression
lr_baseline = LinearRegression()
lr_baseline.fit(X_train_scaled, y_train)

# Make predictions
y_pred_lr_baseline = lr_baseline.predict(X_test_scaled)

# Calculate metrics
mse_lr_baseline = mean_squared_error(y_test, y_pred_lr_baseline)
rmse_lr_baseline = np.sqrt(mse_lr_baseline)
mae_lr_baseline = mean_absolute_error(y_test, y_pred_lr_baseline)
r2_lr_baseline = r2_score(y_test, y_pred_lr_baseline)

# Calculate adjusted R²
n = len(y_test)
p = X_test_scaled.shape[1]
adj_r2_lr_baseline = 1 - (1 - r2_lr_baseline) * (n - 1) / (n - p - 1)

print("=" * 80)
print("BASELINE LINEAR REGRESSION PERFORMANCE")
print("=" * 80)
print(f"Mean Squared Error (MSE):       {mse_lr_baseline:,.2f}")
print(f"Root Mean Squared Error (RMSE): {rmse_lr_baseline:,.2f}")
print(f"Mean Absolute Error (MAE):      {mae_lr_baseline:,.2f}")
print(f"R² Score:                       {r2_lr_baseline:.4f}")
print(f"Adjusted R² Score:              {adj_r2_lr_baseline:.4f}")

print("\nModel Coefficients:")
coef_df = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': lr_baseline.coef_
}).sort_values('Coefficient', ascending=False)
print(coef_df.to_string(index=False))

print(f"\nIntercept: {lr_baseline.intercept_:.4f}")

print("\n" + "=" * 80)
print("COMPARISON: Linear Regression vs XGBoost")
print("=" * 80)
print(f"{'Metric':<20} {'Linear Reg':<15} {'XGBoost':<15} {'Difference':<15}")
print("-" * 80)
print(f"{'RMSE':<20} {rmse_lr_baseline:<15,.2f} {rmse_bayes:<15,.2f} {rmse_lr_baseline-rmse_bayes:<+15,.2f}")
print(f"{'MAE':<20} {mae_lr_baseline:<15,.2f} {mae_bayes:<15,.2f} {mae_lr_baseline-mae_bayes:<+15,.2f}")
print(f"{'R²':<20} {r2_lr_baseline:<15.4f} {r2_bayes:<15.4f} {r2_lr_baseline-r2_bayes:<+15.4f}")
print("=" * 80)

## 7. Linear Regression - Assumption Testing

**CRITICAL**: Linear regression requires five key assumptions to be valid. Violations can lead to:
- Biased coefficient estimates
- Invalid confidence intervals and p-values
- Poor predictions

We'll test each assumption using statistical tests and visualizations.

### The Five Assumptions:
1. **Linearity**: Linear relationship between features and target
2. **Independence**: Residuals are independent (no autocorrelation)
3. **Homoscedasticity**: Constant variance of residuals
4. **Normality**: Residuals are normally distributed
5. **No Multicollinearity**: Features are not highly correlated

Let's examine each in detail...

### 7.1 Assumption 1: Linearity

**Definition**: The relationship between each feature and the target is linear.

**Why it matters**: Linear regression assumes $y = \beta X$. If the true relationship is non-linear (e.g., quadratic, exponential), predictions will be biased.

**Tests:**
- Visual: Scatter plots of each feature vs target
- Visual: Residuals vs Fitted values (should show no pattern)
- Statistical: Pearson correlation (detects linear relationships)

**What to look for:**
✓ Scatter plots show roughly straight-line relationships
✓ Residuals vs Fitted shows random scatter (no curvature)
✗ Curved patterns indicate non-linearity

**Solutions if violated:**
- Transform features (log, sqrt, polynomial)
- Add polynomial features
- Use non-linear models (decision trees, neural networks)

In [ ]:
# Calculate residuals for assumption testing
residuals_lr = y_test - y_pred_lr_baseline

# Linearity Check: Residuals vs Fitted Values
fig_linearity = go.Figure()

fig_linearity.add_trace(go.Scatter(
    x=y_pred_lr_baseline,
    y=residuals_lr,
    mode='markers',
    marker=dict(size=6, color='steelblue', opacity=0.6),
    name='Residuals'
))

# Add zero line
fig_linearity.add_hline(y=0, line_dash="dash", line_color="red", line_width=2)

# Add LOWESS smoothing curve to detect patterns
from scipy.signal import savgol_filter
sorted_indices = np.argsort(y_pred_lr_baseline)
smoothed = savgol_filter(residuals_lr.values[sorted_indices], 
                         window_length=min(51, len(residuals_lr)//2*2-1), 
                         polyorder=3)

fig_linearity.add_trace(go.Scatter(
    x=y_pred_lr_baseline[sorted_indices],
    y=smoothed,
    mode='lines',
    line=dict(color='orange', width=3),
    name='Trend (LOWESS)'
))

fig_linearity.update_layout(
    title='<b>Linearity Check: Residuals vs Fitted Values</b><br>' + 
          '<sub>Random scatter with no pattern = Linearity assumption met</sub>',
    xaxis_title='Fitted Values',
    yaxis_title='Residuals',
    width=900,
    height=500,
    hovermode='closest'
)

fig_linearity.show()

print("\n📊 Linearity Assessment:")
if abs(residuals_lr.mean()) < 0.01 * residuals_lr.std():
    print("  ✓ Residuals centered around zero (good!)")
else:
    print("  ✗ Residuals not centered around zero (potential bias)")

# Check correlation between features and target
print("\n📈 Feature-Target Correlations:")
correlations = []
for col in X.columns:
    corr = np.corrcoef(X[col], y)[0, 1]
    correlations.append((col, corr))

correlations.sort(key=lambda x: abs(x[1]), reverse=True)
for feat, corr in correlations[:5]:
    print(f"  {feat}: {corr:.4f}")

### 7.2 Assumption 2: Independence of Residuals

**Definition**: Residuals (errors) are independent of each other - no autocorrelation.

**Why it matters**: Correlated errors violate the assumption that observations are independent. Common in time series or spatial data.

**Durbin-Watson Test:**
- Tests for first-order autocorrelation
- Range: 0 to 4
- **Interpretation:**
  - DW ≈ 2: No autocorrelation (✓ assumption met)
  - DW < 2: Positive autocorrelation
  - DW > 2: Negative autocorrelation
  - Rule of thumb: 1.5 < DW < 2.5 is acceptable

**What to look for:**
✓ Durbin-Watson close to 2
✓ Residuals plot shows random pattern over index
✗ Patterns in residuals over time/index

**Solutions if violated:**
- Add lagged variables
- Use time series models (ARIMA, VAR)
- Include time-based features

In [ ]:
# Durbin-Watson test for autocorrelation
dw_statistic = durbin_watson(residuals_lr)

print("=" * 80)
print("INDEPENDENCE TEST (Durbin-Watson)")
print("=" * 80)
print(f"Durbin-Watson Statistic: {dw_statistic:.4f}")
print(f"\nInterpretation:")
if 1.5 < dw_statistic < 2.5:
    print(f"  ✓ DW ≈ 2.0: No significant autocorrelation detected")
    print(f"  ✓ Independence assumption is satisfied!")
elif dw_statistic < 1.5:
    print(f"  ⚠ DW < 1.5: Positive autocorrelation detected")
    print(f"  ✗ Independence assumption may be violated")
else:
    print(f"  ⚠ DW > 2.5: Negative autocorrelation detected")
    print(f"  ✗ Independence assumption may be violated")
print("=" * 80)

# Plot residuals over index
fig_independence = go.Figure()

fig_independence.add_trace(go.Scatter(
    x=np.arange(len(residuals_lr)),
    y=residuals_lr.values,
    mode='lines+markers',
    marker=dict(size=4, color='purple'),
    line=dict(color='purple', width=1),
    name='Residuals'
))

fig_independence.add_hline(y=0, line_dash="dash", line_color="red")

fig_independence.update_layout(
    title='<b>Independence Check: Residuals vs Observation Index</b><br>' + 
          '<sub>Should show no pattern - random scatter around zero</sub>',
    xaxis_title='Observation Index',
    yaxis_title='Residuals',
    width=900,
    height=400
)

fig_independence.show()

### 7.3 Assumption 3: Homoscedasticity (Constant Variance)

**Definition**: The variance of residuals is constant across all levels of predicted values.

**Why it matters**: 
- Heteroscedasticity (non-constant variance) leads to inefficient estimates
- Standard errors become unreliable  
- Confidence intervals and hypothesis tests invalid

**Tests:**
- **Visual**: Residuals vs Fitted values (spread should be constant)
- **Scale-Location plot**: √|Standardized Residuals| vs Fitted values
- **Breusch-Pagan test**: Statistical test for heteroscedasticity
  - H₀: Homoscedasticity (constant variance)
  - H₁: Heteroscedasticity (non-constant variance)
  - p-value > 0.05 → Fail to reject H₀ (good!)

**What to look for:**
✓ Constant band width in residual plot
✓ No funnel/cone shape
✓ Breusch-Pagan p-value > 0.05
✗ Expanding/contracting variance

**Solutions if violated:**
- Transform target variable (log, sqrt)
- Use weighted least squares
- Use robust standard errors
- Try different models

In [ ]:
# Breusch-Pagan test for heteroscedasticity
# Prepare data for statsmodels
X_test_with_const = add_constant(X_test_scaled)
bp_test = het_breuschpagan(residuals_lr, X_test_with_const)
bp_lm_statistic, bp_lm_pvalue, bp_f_statistic, bp_f_pvalue = bp_test

print("=" * 80)
print("HOMOSCEDASTICITY TEST (Breusch-Pagan)")
print("=" * 80)
print(f"LM Statistic: {bp_lm_statistic:.4f}")
print(f"LM p-value:   {bp_lm_pvalue:.4f}")
print(f"F-statistic:  {bp_f_statistic:.4f}")
print(f"F p-value:    {bp_f_pvalue:.4f}")
print(f"\nInterpretation (using p-value = 0.05):")
if bp_lm_pvalue > 0.05:
    print(f"  ✓ p-value > 0.05: Fail to reject H₀")
    print(f"  ✓ Homoscedasticity assumption is satisfied!")
else:
    print(f"  ✗ p-value < 0.05: Reject H₀")
    print(f"  ✗ Heteroscedasticity detected - variance is not constant")
print("=" * 80)

# Scale-Location Plot
standardized_residuals = (residuals_lr - residuals_lr.mean()) / residuals_lr.std()
sqrt_abs_std_residuals = np.sqrt(np.abs(standardized_residuals))

fig_homo = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Residuals vs Fitted', 'Scale-Location Plot')
)

# Plot 1: Residuals vs Fitted
fig_homo.add_trace(
    go.Scatter(x=y_pred_lr_baseline, y=residuals_lr,
              mode='markers', marker=dict(size=5, color='teal', opacity=0.6),
              name='Residuals'),
    row=1, col=1
)
fig_homo.add_hline(y=0, line_dash="dash", line_color="red", row=1, col=1)

# Plot 2: Scale-Location
fig_homo.add_trace(
    go.Scatter(x=y_pred_lr_baseline, y=sqrt_abs_std_residuals,
              mode='markers', marker=dict(size=5, color='coral', opacity=0.6),
              name='√|Std Residuals|'),
    row=1, col=2
)

# Add smoothing line to Scale-Location
sorted_idx = np.argsort(y_pred_lr_baseline)
if len(sqrt_abs_std_residuals) > 51:
    smooth_sl = savgol_filter(sqrt_abs_std_residuals.values[sorted_idx], 
                              window_length=51, polyorder=3)
    fig_homo.add_trace(
        go.Scatter(x=y_pred_lr_baseline[sorted_idx], y=smooth_sl,
                  mode='lines', line=dict(color='red', width=2),
                  name='Trend'),
        row=1, col=2
    )

fig_homo.update_xaxes(title_text="Fitted Values", row=1, col=1)
fig_homo.update_xaxes(title_text="Fitted Values", row=1, col=2)
fig_homo.update_yaxes(title_text="Residuals", row=1, col=1)
fig_homo.update_yaxes(title_text="√|Standardized Residuals|", row=1, col=2)

fig_homo.update_layout(
    title_text="<b>Homoscedasticity Check</b><br>" + 
               "<sub>Right plot should show horizontal band (constant spread)</sub>",
    height=450,
    width=1000,
    showlegend=False
)

fig_homo.show()

### 7.4 Assumption 4: Normality of Residuals

**Definition**: Residuals follow a normal (Gaussian) distribution.

**Why it matters**:
- Required for valid hypothesis testing (t-tests, F-tests)
- Confidence intervals rely on normality
- Less critical for large samples (Central Limit Theorem)

**Tests:**
- **Q-Q Plot** (Quantile-Quantile): Residuals vs theoretical normal quantiles
  - Points on diagonal line → Normal distribution
  - Deviations at tails → Heavy/light tails
  
- **Histogram**: Should resemble bell curve

- **Shapiro-Wilk Test**: 
  - H₀: Data is normally distributed
  - p-value > 0.05 → Normal (but sensitive to sample size)

- **Anderson-Darling Test**: More powerful than Shapiro-Wilk

**What to look for:**
✓ Q-Q plot points follow diagonal line
✓ Histogram is bell-shaped
✓ Shapiro-Wilk p-value > 0.05
✗ Skewed distribution, heavy tails

**Solutions if violated:**
- Transform target variable (Box-Cox, Yeo-Johnson)
- Use robust regression
- For large samples (n > 30), less critical due to CLT

In [ ]:
# Normality tests
shapiro_stat, shapiro_pvalue = shapiro(residuals_lr)
anderson_result = anderson(residuals_lr)

print("=" * 80)
print("NORMALITY TESTS")
print("=" * 80)
print("Shapiro-Wilk Test:")
print(f"  Statistic: {shapiro_stat:.4f}")
print(f"  p-value:   {shapiro_pvalue:.4f}")
if shapiro_pvalue > 0.05:
    print(f"  ✓ p-value > 0.05: Residuals are normally distributed")
else:
    print(f"  ✗ p-value < 0.05: Residuals deviate from normality")
    print(f"    (Note: With large samples, often rejects normality even for minor deviations)")

print(f"\nAnderson-Darling Test:")
print(f"  Statistic: {anderson_result.statistic:.4f}")
print(f"  Critical values: {anderson_result.critical_values}")
print(f"  Significance levels: {anderson_result.significance_level}%")
print("=" * 80)

# Create Q-Q plot and histogram
fig_norm = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Q-Q Plot', 'Histogram of Residuals')
)

# Q-Q Plot
theoretical_quantiles = stats.probplot(residuals_lr, dist="norm")[0][0]
sample_quantiles = stats.probplot(residuals_lr, dist="norm")[0][1]

fig_norm.add_trace(
    go.Scatter(x=theoretical_quantiles, y=sample_quantiles,
              mode='markers', marker=dict(size=5, color='darkblue', opacity=0.6),
              name='Residual Quantiles'),
    row=1, col=1
)

# Add diagonal reference line
q_min, q_max = theoretical_quantiles.min(), theoretical_quantiles.max()
fig_norm.add_trace(
    go.Scatter(x=[q_min, q_max], y=[q_min, q_max],
              mode='lines', line=dict(color='red', dash='dash', width=2),
              name='Normal Distribution'),
    row=1, col=1
)

# Histogram with normal curve overlay
fig_norm.add_trace(
    go.Histogram(x=residuals_lr, nbinsx=30, name='Residuals',
                marker_color='lightgreen', opacity=0.7),
    row=1, col=2
)

# Add normal distribution curve
x_norm = np.linspace(residuals_lr.min(), residuals_lr.max(), 100)
y_norm = stats.norm.pdf(x_norm, residuals_lr.mean(), residuals_lr.std())
# Scale to match histogram
y_norm_scaled = y_norm * len(residuals_lr) * (residuals_lr.max() - residuals_lr.min()) / 30

fig_norm.add_trace(
    go.Scatter(x=x_norm, y=y_norm_scaled,
              mode='lines', line=dict(color='red', width=3),
              name='Normal Curve'),
    row=1, col=2
)

fig_norm.update_xaxes(title_text="Theoretical Quantiles", row=1, col=1)
fig_norm.update_yaxes(title_text="Sample Quantiles", row=1, col=1)
fig_norm.update_xaxes(title_text="Residuals", row=1, col=2)
fig_norm.update_yaxes(title_text="Frequency", row=1, col=2)

fig_norm.update_layout(
    title_text="<b>Normality Check</b><br>" +
               "<sub>Q-Q plot points should follow red line; Histogram should match bell curve</sub>",
    height=450,
    width=1000,
    showlegend=True
)

fig_norm.show()

print("\n📊 Residual Distribution Statistics:")
print(f"  Mean: {residuals_lr.mean():.6f} (should be ≈ 0)")
print(f"  Std:  {residuals_lr.std():.4f}")
print(f"  Skewness: {residuals_lr.skew():.4f} (0 = symmetric)")
print(f"  Kurtosis: {residuals_lr.kurtosis():.4f} (0 = normal, >0 = heavy tails)")

### 7.5 Assumption 5: No Multicollinearity

**Definition**: Independent variables (features) are not highly correlated with each other.

**Why it matters**:
- High correlation makes it difficult to isolate individual feature effects
- Coefficient estimates become unstable (high variance)
- Difficult to interpret which feature is truly important
- Model predictions are still accurate, but understanding is compromised

**Variance Inflation Factor (VIF)**:
Measures how much the variance of a coefficient is inflated due to collinearity.

**Formula**: 
$$VIF_i = \frac{1}{1 - R_i^2}$$
where $R_i^2$ is the R² from regressing feature $i$ on all other features.

**Interpretation:**
- **VIF = 1**: No correlation with other features (perfect!)
- **VIF < 5**: Low to moderate correlation (acceptable)
- **VIF 5-10**: Moderate to high correlation (caution!)
- **VIF > 10**: Severe multicollinearity (problematic!)

**What to look for:**
✓ All VIF values < 5
✓ Correlation matrix shows low correlations between features
✗ VIF > 10 for any feature
✗ Correlation coefficients > 0.8 between features

**Solutions if violated**:
- Remove one of the correlated features
- Combine correlated features (PCA, factor analysis)
- Use regularization (Ridge, Lasso, ElasticNet)
- Collect more data with different correlations

In [ ]:
# Calculate VIF for each feature
vif_data = pd.DataFrame()
vif_data["Feature"] = X.columns
vif_data["VIF"] = [variance_inflation_factor(X_train_scaled, i) 
                   for i in range(X_train_scaled.shape[1])]
vif_data = vif_data.sort_values('VIF', ascending=False)

print("=" * 80)
print("MULTICOLLINEARITY TEST (Variance Inflation Factor)")
print("=" * 80)
print(vif_data.to_string(index=False))
print("\nInterpretation:")
print("  VIF < 5:  Low correlation ✓")
print("  VIF 5-10: Moderate correlation ⚠")
print("  VIF > 10: High correlation (multicollinearity issue) ✗")

max_vif = vif_data['VIF'].max()
if max_vif < 5:
    print(f"\n✓ All VIF values < 5: No multicollinearity detected!")
elif max_vif < 10:
    print(f"\n⚠ Some VIF values between 5-10: Moderate multicollinearity")
    print(f"  Features with VIF > 5:")
    high_vif = vif_data[vif_data['VIF'] > 5]
    for _, row in high_vif.iterrows():
        print(f"    {row['Feature']}: VIF = {row['VIF']:.2f}")
else:
    print(f"\n✗ Some VIF values > 10: Severe multicollinearity!")
    print(f"  Features with VIF > 10:")
    severe_vif = vif_data[vif_data['VIF'] > 10]
    for _, row in severe_vif.iterrows():
        print(f"    {row['Feature']}: VIF = {row['VIF']:.2f}")
    print(f"  Consider removing or combining these features")

print("=" * 80)

# Visualize VIF
fig_vif = go.Figure(go.Bar(
    y=vif_data['Feature'],
    x=vif_data['VIF'],
    orientation='h',
    marker=dict(
        color=vif_data['VIF'],
        colorscale=[[0, 'green'], [0.5, 'yellow'], [1, 'red']],
        cmin=0,
        cmax=10,
        showscale=True,
        colorbar=dict(title="VIF Value")
    ),
    text=vif_data['VIF'].round(2),
    textposition='outside'
))

# Add reference lines
fig_vif.add_vline(x=5, line_dash="dash", line_color="orange", 
                  annotation_text="Moderate Threshold (VIF=5)")
fig_vif.add_vline(x=10, line_dash="dash", line_color="red", 
                  annotation_text="High Threshold (VIF=10)")

fig_vif.update_layout(
    title='<b>Multicollinearity Check: Variance Inflation Factor (VIF)</b><br>' + 
          '<sub>Lower is better - VIF < 5 is ideal</sub>',
    xaxis_title='VIF Value',
    yaxis_title='Features',
    width=900,
    height=max(400, len(X.columns) * 30),
    showlegend=False
)

fig_vif.show()

## 8. Linear Regression - Regularization and Hyperparameter Tuning

### Regularization in Linear Regression

While basic Linear Regression has no traditional "hyperparameters" to tune, we can add **regularization** to prevent overfitting:

**Ridge Regression (L2 Regularization):**
$$\text{minimize} \sum (y - \hat{y})^2 + \alpha \sum \beta^2$$
- Adds penalty on coefficient magnitudes (squared)
- Shrinks coefficients toward zero (but never exactly zero)
- Handles multicollinearity well
- All features retained

**Lasso Regression (L1 Regularization):**
$$\text{minimize} \sum (y - \hat{y})^2 + \alpha \sum |\beta|$$
- Adds penalty on absolute coefficient values
- Can shrink coefficients to exactly zero → **Feature selection**
- Useful when many irrelevant features

**ElasticNet (L1 + L2):**
$$\text{minimize} \sum (y - \hat{y})^2 + \alpha_1 \sum |\beta| + \alpha_2 \sum \beta^2$$
- Combines both L1 and L2 penalties
- Balances feature selection and coefficient shrinkage
- Parameter `l1_ratio` controls mix (0=Ridge, 1=Lasso)

**Hyperparameter α (alpha)**:
- Controls strength of regularization
- α = 0: No regularization (standard linear regression)
- α → ∞: All coefficients → 0

### 8.1 Grid Search for Regularization

In [ ]:
# Grid Search for Ridge Regression
param_grid_ridge = {
    'alpha': [0.001, 0.01, 0.1, 1, 10, 100, 1000]
}

grid_ridge = GridSearchCV(
    Ridge(random_state=42),
    param_grid_ridge,
    scoring='neg_mean_squared_error',
    cv=5,
    verbose=1,
    n_jobs=-1
)

print("=" * 80)
print("GRID SEARCH: RIDGE REGRESSION")
print("=" * 80)
grid_ridge.fit(X_train_scaled, y_train)
print(f"✓ Best alpha: {grid_ridge.best_params_['alpha']}")
print(f"✓ Best CV RMSE: {np.sqrt(-grid_ridge.best_score_):,.2f}")

# Evaluate on test set
y_pred_ridge = grid_ridge.best_estimator_.predict(X_test_scaled)
r2_ridge = r2_score(y_test, y_pred_ridge)
rmse_ridge = np.sqrt(mean_squared_error(y_test, y_pred_ridge))
print(f"✓ Test RMSE: {rmse_ridge:,.2f}")
print(f"✓ Test R²: {r2_ridge:.4f}")

# Grid Search for Lasso Regression
param_grid_lasso = {
    'alpha': [0.001, 0.01, 0.1, 1, 10, 100]
}

grid_lasso = GridSearchCV(
    Lasso(random_state=42, max_iter=10000),
    param_grid_lasso,
    scoring='neg_mean_squared_error',
    cv=5,
    verbose=1,
    n_jobs=-1
)

print("\n" + "=" * 80)
print("GRID SEARCH: LASSO REGRESSION")
print("=" * 80)
grid_lasso.fit(X_train_scaled, y_train)
print(f"✓ Best alpha: {grid_lasso.best_params_['alpha']}")
print(f"✓ Best CV RMSE: {np.sqrt(-grid_lasso.best_score_):,.2f}")

y_pred_lasso = grid_lasso.best_estimator_.predict(X_test_scaled)
r2_lasso = r2_score(y_test, y_pred_lasso)
rmse_lasso = np.sqrt(mean_squared_error(y_test, y_pred_lasso))
print(f"✓ Test RMSE: {rmse_lasso:,.2f}")
print(f"✓ Test R²: {r2_lasso:.4f}")

# Count non-zero coefficients (feature selection)
non_zero_coefs = np.sum(grid_lasso.best_estimator_.coef_ != 0)
print(f"✓ Non-zero coefficients: {non_zero_coefs}/{len(X.columns)} (Lasso does feature selection!)")

# Grid Search for ElasticNet
param_grid_elastic = {
    'alpha': [0.001, 0.01, 0.1, 1, 10],
    'l1_ratio': [0.1, 0.3, 0.5, 0.7, 0.9]  # Mix of L1 and L2
}

grid_elastic = GridSearchCV(
    ElasticNet(random_state=42, max_iter=10000),
    param_grid_elastic,
    scoring='neg_mean_squared_error',
    cv=5,
    verbose=1,
    n_jobs=-1
)

print("\n" + "=" * 80)
print("GRID SEARCH: ELASTICNET REGRESSION")
print("=" * 80)
grid_elastic.fit(X_train_scaled, y_train)
print(f"✓ Best alpha: {grid_elastic.best_params_['alpha']}")
print(f"✓ Best l1_ratio: {grid_elastic.best_params_['l1_ratio']}")
print(f"✓ Best CV RMSE: {np.sqrt(-grid_elastic.best_score_):,.2f}")

y_pred_elastic = grid_elastic.best_estimator_.predict(X_test_scaled)
r2_elastic = r2_score(y_test, y_pred_elastic)
rmse_elastic = np.sqrt(mean_squared_error(y_test, y_pred_elastic))
print(f"✓ Test RMSE: {rmse_elastic:,.2f}")
print(f"✓ Test R²: {r2_elastic:.4f}")

print("\n" + "=" * 80)
print("REGULARIZATION COMPARISON")
print("=" * 80)
print(f"{'Model':<20} {'RMSE':<15} {'R²':<15}")
print("-" * 80)
print(f"{'Baseline':<20} {rmse_lr_baseline:<15,.2f} {r2_lr_baseline:<15.4f}")
print(f"{'Ridge (Best)':<20} {rmse_ridge:<15,.2f} {r2_ridge:<15.4f}")
print(f"{'Lasso (Best)':<20} {rmse_lasso:<15,.2f} {r2_lasso:<15.4f}")
print(f"{'ElasticNet (Best)':<20} {rmse_elastic:<15,.2f} {r2_elastic:<15.4f}")
print("=" * 80)

### 8.2 Random Search & Bayesian Optimization for Regularization

We'll apply the same optimization techniques to Ridge/Lasso/ElasticNet.

In [ ]:
# Bayesian Optimization for Ridge Regression
search_space_ridge = {
    'alpha': Real(0.0001, 1000, prior='log-uniform', name='alpha')
}

bayes_ridge = BayesSearchCV(
    Ridge(random_state=42),
    search_space_ridge,
    n_iter=30,
    scoring='neg_mean_squared_error',
    cv=5,
    random_state=42,
    n_jobs=-1
)

print("=" * 80)
print("BAYESIAN OPTIMIZATION: RIDGE REGRESSION")
print("=" * 80)
bayes_ridge.fit(X_train_scaled, y_train)
print(f"✓ Best alpha: {bayes_ridge.best_params_['alpha']:.6f}")

y_pred_bayes_ridge = bayes_ridge.best_estimator_.predict(X_test_scaled)
r2_bayes_ridge = r2_score(y_test, y_pred_bayes_ridge)
rmse_bayes_ridge = np.sqrt(mean_squared_error(y_test, y_pred_bayes_ridge))
mae_bayes_ridge = mean_absolute_error(y_test, y_pred_bayes_ridge)

print(f"✓ Test RMSE: {rmse_bayes_ridge:,.2f}")
print(f"✓ Test MAE:  {mae_bayes_ridge:,.2f}")
print(f"✓ Test R²:   {r2_bayes_ridge:.4f}")
print("=" * 80)

# Store best linear regression model
lr_best_model = bayes_ridge.best_estimator_
print(f"\n✓ Best Linear Regression model (Ridge) saved as 'lr_best_model'")

## 9. Gradient Descent Visualization

**Gradient Descent** is an iterative optimization algorithm used to find the minimum of a function. For linear regression, it minimizes the cost function (MSE).

### Algorithm:
1. **Initialize** parameters θ (coefficients) randomly
2. **Calculate** cost function: $J(\theta) = \frac{1}{2m} \sum (h_\theta(x) - y)^2$
3. **Compute** gradient: $\frac{\partial J}{\partial \theta} = \frac{1}{m} X^T (X\theta - y)$
4. **Update** parameters: $\theta := \theta - \alpha \nabla J(\theta)$
5. **Repeat** until convergence

### Key Parameter: Learning Rate (α)
- Too small → Slow convergence
- Too large → May overshoot minimum
- Just right → Efficient convergence

### Types:
- **Batch GD**: Uses all training data per iteration (what we'll implement)
- **Stochastic GD**: Uses one sample per iteration
- **Mini-batch GD**: Uses small batch per iteration

Let's implement and visualize gradient descent for a simple 2D case!

In [ ]:
# Implement Gradient Descent from scratch
def gradient_descent(X, y, learning_rate=0.01, epochs=1000):
    """
    Gradient Descent implementation for Linear Regression
    
    Parameters:
    - X: Feature matrix (m x n)
    - y: Target vector (m x 1)
    - learning_rate: Step size (alpha)
    - epochs: Number of iterations
    
    Returns:
    - theta: Learned parameters
    - cost_history: Cost at each iteration
    - theta_history: Parameter values at each iteration
    """
    m, n = X.shape
    theta = np.zeros(n)  # Initialize parameters
    cost_history = []
    theta_history = []
    
    for epoch in range(epochs):
        # Predictions
        predictions = X.dot(theta)
        
        # Calculate cost (MSE)
        cost = (1/(2*m)) * np.sum((predictions - y)**2)
        cost_history.append(cost)
        theta_history.append(theta.copy())
        
        # Calculate gradient
        gradient = (1/m) * X.T.dot(predictions - y)
        
        # Update parameters
        theta = theta - learning_rate * gradient
    
    return theta, cost_history, theta_history

# Use a simple 2-feature subset for visualization
# Select two features with highest correlation to target
top_2_features = correlations[:2]
X_simple = X_train[[feat for feat, _ in top_2_features]].values
X_simple_scaled = scaler.fit_transform(X_simple)

# Add bias term
X_simple_with_bias = np.c_[np.ones(X_simple_scaled.shape[0]), X_simple_scaled]

# Run gradient descent with different learning rates
learning_rates = [0.001, 0.01, 0.1]
results = {}

print("=" * 80)
print("GRADIENT DESCENT TRAINING")
print("=" * 80)

for lr in learning_rates:
    theta, cost_hist, theta_hist = gradient_descent(
        X_simple_with_bias, 
        y_train.values, 
        learning_rate=lr, 
        epochs=500
    )
    results[lr] = {'theta': theta, 'cost_history': cost_hist, 'theta_history': theta_hist}
    final_cost = cost_hist[-1]
    print(f"Learning Rate: {lr:.3f}")
    print(f"  Final Cost: {final_cost:,.2f}")
    print(f"  Parameters: {theta}")
    print()

print("=" * 80)

# Visualize cost function convergence
fig_gd_cost = go.Figure()

for lr in learning_rates:
    fig_gd_cost.add_trace(go.Scatter(
        x=np.arange(len(results[lr]['cost_history'])),
        y=results[lr]['cost_history'],
        mode='lines',
        name=f'α = {lr}',
        line=dict(width=2)
    ))

fig_gd_cost.update_layout(
    title='<b>Gradient Descent: Cost Function Convergence</b><br>' + 
          '<sub>Shows how cost decreases over iterations for different learning rates</sub>',
    xaxis_title='Epoch (Iteration)',
    yaxis_title='Cost (MSE)',
    xaxis_type='log',
    yaxis_type='log',
    width=900,
    height=500,
    hovermode='x unified'
)

fig_gd_cost.show()

print("\n📊 Observations:")
print("  - α too small (0.001): Slow convergence")
print("  - α too large (0.1): May oscillate")
print("  - α optimal (0.01): Smooth, fast convergence")

## 10. Linear Regression - Comprehensive Visualizations

Let's create interactive Plotly visualizations for Linear Regression performance.

In [ ]:
# 1. Actual vs Predicted for Linear Regression
fig_lr_pred = go.Figure()

fig_lr_pred.add_trace(go.Scatter(
    x=y_test,
    y=y_pred_bayes_ridge,
    mode='markers',
    name='Predictions',
    marker=dict(size=8, color='forestgreen', opacity=0.6),
    text=[f'Actual: {a:.2f}<br>Predicted: {p:.2f}<br>Error: {a-p:.2f}' 
          for a, p in zip(y_test, y_pred_bayes_ridge)],
    hovertemplate='%{text}<extra></extra>'
))

# Perfect prediction line
min_val = min(y_test.min(), y_pred_bayes_ridge.min())
max_val = max(y_test.max(), y_pred_bayes_ridge.max())
fig_lr_pred.add_trace(go.Scatter(
    x=[min_val, max_val],
    y=[min_val, max_val],
    mode='lines',
    name='Perfect Prediction',
    line=dict(color='red', dash='dash', width=2)
))

fig_lr_pred.update_layout(
    title=f'<b>Linear Regression: Actual vs Predicted Values</b><br><sub>R² = {r2_bayes_ridge:.4f}, RMSE = {rmse_bayes_ridge:,.2f}</sub>',
    xaxis_title='Actual Values',
    yaxis_title='Predicted Values',
    width=800,
    height=600
)

fig_lr_pred.show()

# 2. Coefficient Plot
coef_df_ridge = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': lr_best_model.coef_
}).sort_values('Coefficient')

fig_coef = go.Figure(go.Bar(
    y=coef_df_ridge['Feature'],
    x=coef_df_ridge['Coefficient'],
    orientation='h',
    marker=dict(
        color=coef_df_ridge['Coefficient'],
        colorscale='RdBu',
        cmid=0,
        showscale=True
    ),
    text=coef_df_ridge['Coefficient'].round(4),
    textposition='outside'
))

fig_coef.add_vline(x=0, line_dash="dash", line_color="black", line_width=2)

fig_coef.update_layout(
    title='<b>Linear Regression: Feature Coefficients</b><br>' + 
          '<sub>Positive = increases target, Negative = decreases target</sub>',
    xaxis_title='Coefficient Value',
    yaxis_title='Features',
    width=800,
    height=max(400, len(X.columns) * 25)
)

fig_coef.show()

print("\n📊 Coefficient Interpretation:")
print(f"Intercept: {lr_best_model.intercept_:.4f}")
print("\nTop 3 Positive Coefficients (increase target):")
for _, row in coef_df_ridge.tail(3).iterrows():
    print(f"  {row['Feature']}: {row['Coefficient']:+.4f}")
print("\nTop 3 Negative Coefficients (decrease target):")
for _, row in coef_df_ridge.head(3).iterrows():
    print(f"  {row['Feature']}: {row['Coefficient']:+.4f}")

## 11. Linear Regression - Model Persistence

Save the best Linear Regression model using pickle and joblib.

In [ ]:
# Prepare metadata for Linear Regression
lr_metadata = {
    'model_name': 'Ridge Regression',
    'created_date': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'hyperparameters': lr_best_model.get_params(),
    'feature_names': list(X.columns),
    'coefficients': dict(zip(X.columns, lr_best_model.coef_)),
    'intercept': lr_best_model.intercept_,
    'metrics': {
        'RMSE': rmse_bayes_ridge,
        'MAE': mae_bayes_ridge,
        'R2': r2_bayes_ridge
    },
    'tuning_method': 'Bayesian Optimization',
    'scaler': 'StandardScaler',
    'train_size': len(X_train),
    'test_size': len(X_test)
}

print("=" * 80)
print("SAVING LINEAR REGRESSION MODEL")
print("=" * 80)

# Save model with pickle
lr_pickle_filename = 'models/linear_regression_model.pkl'
with open(lr_pickle_filename, 'wb') as file:
    pickle.dump(lr_best_model, file)
print(f"✓ Model saved with pickle: {lr_pickle_filename}")
print(f"  File size: {os.path.getsize(lr_pickle_filename) / 1024:.2f} KB")

# Save model with joblib
lr_joblib_filename = 'models/linear_regression_model.joblib'
joblib.dump(lr_best_model, lr_joblib_filename, compress=3)
print(f"✓ Model saved with joblib: {lr_joblib_filename}")
print(f"  File size: {os.path.getsize(lr_joblib_filename) / 1024:.2f} KB")

# Save scaler (IMPORTANT for deployment!)
scaler_filename = 'models/scaler.joblib'
joblib.dump(scaler, scaler_filename)
print(f"✓ Scaler saved: {scaler_filename}")

# Save metadata
lr_metadata_filename = 'models/linear_regression_metadata.pkl'
with open(lr_metadata_filename, 'wb') as file:
    pickle.dump(lr_metadata, file)
print(f"✓ Metadata saved: {lr_metadata_filename}")

print("\n" + "=" * 80)
print("LOADING AND TESTING LINEAR REGRESSION MODEL")
print("=" * 80)

# Load and test
lr_loaded = joblib.load(lr_joblib_filename)
scaler_loaded = joblib.load(scaler_filename)

# Make predictions with loaded model
X_test_sample_scaled = scaler_loaded.transform(X_test[:5])
y_pred_loaded = lr_loaded.predict(X_test_sample_scaled)

print(f"✓ Model and scaler loaded successfully")
print(f"\n🔍 Verification (first 5 predictions):")
print(f"Original:    {y_pred_bayes_ridge[:5]}")
print(f"From Loaded: {y_pred_loaded}")

if np.allclose(y_pred_bayes_ridge[:5], y_pred_loaded):
    print("\n✅ SUCCESS: Loaded model produces identical predictions!")
else:
    print("\n⚠ WARNING: Predictions differ!")

print("=" * 80)

---
# 🏆 FINAL COMPARISON: XGBoost vs Linear Regression
---

## 12. Comprehensive Model Comparison

Let's compare both models across multiple dimensions: performance, complexity, interpretability, and computational cost.

In [ ]:
# Final comparison metrics
comparison_df = pd.DataFrame({
    'Metric': ['RMSE', 'MAE', 'R² Score', 'Adjusted R²'],
    'XGBoost': [
        rmse_bayes,
        mae_bayes,
        r2_bayes,
        1 - (1 - r2_bayes) * (len(y_test) - 1) / (len(y_test) - X_test.shape[1] - 1)
    ],
    'Linear Regression': [
        rmse_bayes_ridge,
        mae_bayes_ridge,
        r2_bayes_ridge,
        1 - (1 - r2_bayes_ridge) * (len(y_test) - 1) / (len(y_test) - X_test.shape[1] - 1)
    ]
})

comparison_df['Difference'] = comparison_df['XGBoost'] - comparison_df['Linear Regression']
comparison_df['Winner'] = comparison_df.apply(
    lambda row: 'XGBoost' if (row['Metric'] in ['R² Score', 'Adjusted R²'] and row['Difference'] > 0) 
                or (row['Metric'] in ['RMSE', 'MAE'] and row['Difference'] < 0)
                else 'Linear Regression' if row['Difference'] != 0 else 'Tie',
    axis=1
)

print("=" * 100)
print("FINAL MODEL COMPARISON")
print("=" * 100)
print(comparison_df.to_string(index=False))
print("=" * 100)

# Visualize comparison
fig_comparison = make_subplots(
    rows=2, cols=2,
    subplot_titles=('RMSE Comparison', 'MAE Comparison', 
                   'R² Score Comparison', 'Predictions Overlay'),
    specs=[[{'type': 'bar'}, {'type': 'bar'}],
           [{'type': 'bar'}, {'type': 'scatter'}]]
)

# RMSE
fig_comparison.add_trace(
    go.Bar(x=['XGBoost', 'Linear Reg'], 
           y=[rmse_bayes, rmse_bayes_ridge],
           marker_color=['royalblue', 'forestgreen'],
           text=[f'{rmse_bayes:,.2f}', f'{rmse_bayes_ridge:,.2f}'],
           textposition='outside'),
    row=1, col=1
)

# MAE
fig_comparison.add_trace(
    go.Bar(x=['XGBoost', 'Linear Reg'],
           y=[mae_bayes, mae_bayes_ridge],
           marker_color=['royalblue', 'forestgreen'],
           text=[f'{mae_bayes:,.2f}', f'{mae_bayes_ridge:,.2f}'],
           textposition='outside'),
    row=1, col=2
)

# R²
fig_comparison.add_trace(
    go.Bar(x=['XGBoost', 'Linear Reg'],
           y=[r2_bayes, r2_bayes_ridge],
           marker_color=['royalblue', 'forestgreen'],
           text=[f'{r2_bayes:.4f}', f'{r2_bayes_ridge:.4f}'],
           textposition='outside'),
    row=2, col=1
)

# Predictions Overlay
fig_comparison.add_trace(
    go.Scatter(x=y_test, y=y_pred_bayes,
              mode='markers', name='XGBoost',
              marker=dict(size=5, color='royalblue', opacity=0.5)),
    row=2, col=2
)
fig_comparison.add_trace(
    go.Scatter(x=y_test, y=y_pred_bayes_ridge,
              mode='markers', name='Linear Reg',
              marker=dict(size=5, color='forestgreen', opacity=0.5)),
    row=2, col=2
)

# Perfect prediction line
min_val = y_test.min()
max_val = y_test.max()
fig_comparison.add_trace(
    go.Scatter(x=[min_val, max_val], y=[min_val, max_val],
              mode='lines', name='Perfect',
              line=dict(color='red', dash='dash')),
    row=2, col=2
)

fig_comparison.update_yaxes(title_text="RMSE", row=1, col=1)
fig_comparison.update_yaxes(title_text="MAE", row=1, col=2)
fig_comparison.update_yaxes(title_text="R²", row=2, col=1)
fig_comparison.update_xaxes(title_text="Actual", row=2, col=2)
fig_comparison.update_yaxes(title_text="Predicted", row=2, col=2)

fig_comparison.update_layout(
    title_text="<b>XGBoost vs Linear Regression: Performance Comparison</b>",
    height=800,
    width=1200,
    showlegend=True
)

fig_comparison.show()

# Summary Analysis
print("\n" + "=" * 100)
print("KEY INSIGHTS & RECOMMENDATIONS")
print("=" * 100)

if r2_bayes > r2_bayes_ridge:
    winner = "XGBoost"
    improvement = ((r2_bayes - r2_bayes_ridge) / r2_bayes_ridge) * 100
    print(f"🏆 WINNER: XGBoost")
    print(f"   - Better R² by {improvement:.2f}%")
    print(f"   - RMSE: {rmse_bayes:,.2f} vs {rmse_bayes_ridge:,.2f}")
else:
    winner = "Linear Regression"
    improvement = ((r2_bayes_ridge - r2_bayes) / r2_bayes) * 100
    print(f"🏆 WINNER: Linear Regression")
    print(f"   - Better R² by {improvement:.2f}%")
    print(f"   - RMSE: {rmse_bayes_ridge:,.2f} vs {rmse_bayes:,.2f}")

print(f"\n📊 Model Characteristics:")
print(f"   XGBoost:")
print(f"     ✓ Can capture non-linear relationships")
print(f"     ✓ Robust to outliers")
print(f"     ✗ Less interpretable (black-box)")
print(f"     ✗ More complex, requires hyperparameter tuning")

print(f"\n   Linear Regression:")
print(f"     ✓ Highly interpretable (clear coefficients)")
print(f"     ✓ Fast training and prediction")
print(f"     ✓ Statistical inference (p-values, confidence intervals)")
print(f"     ✗ Assumes linear relationships")
print(f"     ✗ Sensitive to assumption violations")

print(f"\n🎯 RECOMMENDATION:")
if winner == "XGBoost":
    print(f"   Use XGBoost for:")
    print(f"     - Production prediction tasks (accuracy is priority)")
    print(f"     - Complex non-linear data")
    print(f"     - When interpretability is less critical")
    print(f"\n   Use Linear Regression for:")
    print(f"     - Exploratory analysis and understanding relationships")
    print(f"     - When you need to explain coefficients to stakeholders")
    print(f"     - Baseline model for comparison")
else:
    print(f"   Linear Regression performs well here! Consider using it for:")
    print(f"     - Simpler, more interpretable solution")
    print(f"     - Faster deployment")
    print(f"     - Statistical hypothesis testing")
    print(f"\n   However, still consider XGBoost if:")
    print(f"     - You need slightly better accuracy")
    print(f"     - Data has non-linear patterns")

print("=" * 100)

## 13. Conclusion & Key Takeaways

### 🎓 What We Learned:

#### 1. **Data Preprocessing**
- ✓ Handled missing values
- ✓ Applied feature scaling (StandardScaler for Linear Regression)
- ✓ Created multiple train-test splits (70-30, 80-20, 90-10)

#### 2. **XGBoost Regressor**
- ✓ Gradient boosting algorithm with sequential tree building
- ✓ Three hyperparameter tuning methods:
  - **Grid Search**: Exhaustive but slow (81 combinations)
  - **Random Search**: Efficient sampling (50 iterations)
  - **Bayesian Optimization**: Intelligent search (30 iterations)
- ✓ Feature importance analysis
- ✓ Model persistence with pickle and joblib

#### 3. **Linear Regression**
- ✓ Mathematical foundation: OLS method
- ✓ **Five Critical Assumptions Tested:**
  1. Linearity (Residuals vs Fitted plot)
  2. Independence (Durbin-Watson test)
  3. Homoscedasticity (Breusch-Pagan test, Scale-Location plot)
  4. Normality (Q-Q plot, Shapiro-Wilk test)
  5. No Multicollinearity (VIF analysis)
- ✓ Regularization techniques (Ridge, Lasso, ElasticNet)
- ✓ Gradient descent implementation and visualization
- ✓ Coefficient interpretation

#### 4. **Hyperparameter Tuning Techniques Comparison**

| Method | Iterations | Time | Best Use Case |
|--------|-----------|------|---------------|
| **Grid Search** | All combinations | Slow | Final tuning, small space |
| **Random Search** | Fixed n_iter | Fast | Large space, exploration |
| **Bayesian Optimization** | Adaptive | Efficient | Expensive models, limited budget |

#### 5. **Model Selection Insights**

**When to use XGBoost:**
- Non-linear relationships in data
- Accuracy is the top priority
- Have sufficient data (1000+ samples)
- Can afford longer training time
- Interpretability less important

**When to use Linear Regression:**
- Need interpretable coefficients
- Linear relationships suspected
- Statistical inference required (p-values, confidence intervals)
- Fast training/prediction needed
- Explanation to non-technical stakeholders

#### 6. **Best Practices Applied**
✅ Split data BEFORE scaling (prevent data leakage)
✅ Use cross-validation for robust evaluation
✅ Test multiple hyperparameter tuning methods
✅ Save models AND preprocessing objects (scaler)
✅ Verify assumption for Linear Regression
✅ Compare multiple models
✅ Document model metadata

---

### 📁 Saved Artifacts:
- `models/xgboost_model.joblib` - Best XGBoost model
- `models/linear_regression_model.joblib` - Best Linear Regression model
- `models/scaler.joblib` - StandardScaler for preprocessing
- `models/*_metadata.pkl` - Model metadata and hyperparameters

---

### 🚀 Next Steps:
1. **Feature Engineering**: Create polynomial features, interactions
2. **Ensemble Methods**: Stack XGBoost + Linear Regression
3. **Advanced Tuning**: Use Optuna or Hyperopt for more sophisticated optimization
4. **Deployment**: Create REST API with FastAPI/Flask
5. **Monitoring**: Track model performance over time
6. **A/B Testing**: Test models in production environment

---

### 📚 References & Further Reading:
- **XGBoost**: Chen & Guestrin (2016) - "XGBoost: A Scalable Tree Boosting System"
- **Bayesian Optimization**: Bergstra & Bengio (2012) - "Random Search for Hyper-Parameter Optimization"
- **Linear Regression**: James et al. - "An Introduction to Statistical Learning"
- **Gradient Descent**: Ruder (2016) - "An overview of gradient descent optimization algorithms"

---

**Notebook Created**: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

**Author**: Machine Learning Practitioner

**Purpose**: Educational demonstration of regression techniques with comprehensive explanations

---

### ⭐ Thank you for working through this comprehensive regression analysis notebook!